# Solución proyecto

## Datos de entrada



![](project_scheme_1motor.png)


In [56]:
import numpy as np
np.set_printoptions(precision=2, suppress=True) # Para mostrar 2 decimales

# Esquema físico (m)
A = np.array([60, 45, 65, 50, 40])
B = np.array([18, 25, 20, 15, 22])

# Placa del motor
P_mec = np.array([5, 7.5, 10, 5.5, 8])           # Potencia mecánica (kW)
V_delta = np.array([220, 220, 220, 220, 220]) # Tensión en triángulo (V)
V_estrella = np.array([380, 380, 380, 380, 380]) # Tensión en estrella (V)
I_delta = np.array([9.6, 14.5, 18.2, 13.8, 15.3]) # Corriente (A)
I_estrella = np.array([5.5, 8.4, 10.5, 8.0, 8.8]) # Corriente (A)
f = np.array([50, 50, 50, 50, 50])           # Frecuencia (Hz)
cos_phi = np.array([0.85, 0.87, 0.88, 0.84, 0.86]) # Factor de potencia
η = np.array([0.88, 0.89, 0.90, 0.87, 0.89]) # Eficiencia (adimensional)
n = np.array([1440, 1450, 1460, 1420, 1440]) # Velocidad (rpm)

# Red eléctrica
f_red = np.array([50, 50, 50, 50, 50])       # Frecuencia (Hz)
V_red_fase = np.array([220, 220, 220, 220, 220]) # Tensión fase-neutro (V)
V_red_linea = np.array([380, 380, 380, 380, 380]) # Tensión entre fases (V)

## Etapa 3: Corriente de proyecto y dimensionamiento del conductor

### 1. Cálculo de la corriente de proyecto $I_B$

In [57]:
P_elec = (P_mec / η) * 1000 # Potencia eléctrica activa en W
print(f"P_elec = {P_elec} (W)")
In_motor = P_elec / ( 3**(1/3) * V_estrella * cos_phi)
print(f"In_motor = {In_motor} (Amperes)")
I_B = 1.25 * In_motor
print(f"I_B = {I_B} (Amperes)")

P_elec = [ 5681.82  8426.97 11111.11  6321.84  8988.76] (W)
In_motor = [12.2  17.67 23.04 13.73 19.07] (Amperes)
I_B = [15.25 22.09 28.8  17.17 23.84] (Amperes)


### 2. Dimensionamiento del conductor por capacidad de corriente $I_z$

Se adoptan los siguientes valores:

In [58]:
S = np.array([2.5, 2.5, 2.5, 2.5, 2.5]) # Sección en mm2
I_z = np.array([29, 29, 29, 29, 29]) # Corriente max admisible del conductor

## Etapa 4: Seleccionar y coordinar las protecciones

### 1. Selección del dispositivo de protección

Seleccionar un guardamotor cuya corriente nominal $I_n$ se pueda regular a una corriente $I_r$ lo más cercana posible a la corriente nominal del motor

In [59]:
I_r = In_motor
print(f"I_r = {I_r} (Amperes)")

I_r = [12.2  17.67 23.04 13.73 19.07] (Amperes)


### 2. Verificación contra sobre cargas

Estamos suponiendo que se seleccionó un guardamotor que puede se puede regular de modo que si $I_r$ sea igual a la $I_N$ del motor.

#### Primera condición: $I_r \le I_B \le I_Z$.

In [60]:
print(f"I_r <= I_B <= I_z: {(I_r <= I_B) & (I_B <= I_z)}") # & es And elemento a elemento

I_r <= I_B <= I_z: [ True  True  True  True  True]


#### Segunda condición: $I_2 \le 1,45 \cdot I_Z$

In [61]:
I_2 = I_r * 1.3
print(f"I_2 = {I_2} (Amperes)")
print(f"1.45 * I_z = {1.45 * I_z} (Amperes)")
print(f"I_2 <= 1.45 I_z: {I_2 <= 1.45 * I_z}")

I_2 = [15.86 22.98 29.95 17.85 24.79] (Amperes)
1.45 * I_z = [42.05 42.05 42.05 42.05 42.05] (Amperes)
I_2 <= 1.45 I_z: [ True  True  True  True  True]


### 3. Verificación contra cortocircuitos

#### Corriente máxima de cortocircuito

In [62]:
ρ = 0.018 # En frío en [(Ω ⋅ mm2)/m]
R_cable_a = (ρ * (2 * A)) / S
print(f"R cable (tramo A): {R_cable_a} Ω")
U_L = 380 # V
c = 1 # factor que considera la caída de tensión al momento de la falla
Ik_max = (c * U_L) / (3**(1/2) * R_cable_a)
print(f"Corriente max de cortocircuito: {Ik_max} A")

R cable: [0.86 0.65 0.94 0.72 0.58] Ω
Corriente max de cortocircuito: [253.93 338.57 234.39 304.71 380.89] A


#### Corriente mínima de cortocircuito

In [64]:
ρ = 0.018 # En frío en [(Ω ⋅ mm2)/m]
R_cable_ab = (ρ * (2 * (A+B))) / S
print(f"R cable (tramos A y B): {R_cable_ab} Ω")
Ik_min = (c * U_L) / (3**(1/2) * R_cable_ab)
print(f"Corriente min de cortocircuito: {Ik_min} A")

R cable (tramos A y B): [1.12 1.01 1.22 0.94 0.89] Ω
Corriente min de cortocircuito: [195.33 217.65 179.24 234.39 245.74] A


#### Verificaciones

In [67]:
print(f"Poder de corte Icu del guardamotor >= que {Ik_max} A")
print(f"Corriente de disparo magnética del guardamotor < que {Ik_min} A")

Poder de corte Icu del guardamotor >= que [253.93 338.57 234.39 304.71 380.89] A
Corriente de disparo magnética del guardamotor < que [195.33 217.65 179.24 234.39 245.74] A


### 4. Verificación por caída de tensión

Se usa el método del gradiente.

####  Régimen permanente